# Density-Based Spatial Clustering of Applications with Noise (DBSCAN)

DBSCAN is a powerful unsupervised clustering algorithm that groups points together based on their local density. Unlike K-Means, which assumes clusters are spherical and requires you to pre-specify the number of clusters, DBSCAN can find arbitrarily shaped clusters and automatically identify outliers (noise).

### Core Concepts of DBSCAN:
1. **Epsilon ($\epsilon$)**: The maximum distance between two samples for one to be considered as in the neighborhood of the other (the search radius).
2. **MinPts (Minimum Points)**: The minimum number of points required within the $\epsilon$-neighborhood to form a dense region (a core point).

### Point Classification:
- **Core Point**: A point that has at least `MinPts` within its $\epsilon$-neighborhood (including itself).
- **Border Point**: A point that has fewer than `MinPts` within its $\epsilon$-neighborhood, but lies within the neighborhood of a core point.
- **Noise Point (Outlier)**: A point that is neither a core point nor a border point.

### Why Use DBSCAN?
- **No predefined cluster count**: DBSCAN automatically determines the number of clusters.
- **Arbitrary shapes**: It can easily identify non-spherical clusters (e.g., moons, circles).
- **Robust to noise**: Outliers are identified and labeled as a separate group (`-1`).


## 1. Import Libraries

Let's import the necessary Python packages for loading datasets, standardizing features, training K-Means and DBSCAN models, and plotting results.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True


## 2. Generate and Visualize Synthetic Datasets

Since DBSCAN is designed to excel on non-linearly separated clusters, we'll generate two datasets to compare performance:
1. **Moons Dataset**: Non-linear, crescent-shaped structures that K-Means struggles with.
2. **Blobs Dataset**: Spherical clusters that are easily separated by linear decision boundaries.


In [ ]:
# Generate crescent/moon-shaped dataset (non-linear)
X_moons, y_moons = datasets.make_moons(n_samples=300, noise=0.07, random_state=42)

# Generate simple blob dataset (spherical)
X_blobs, y_blobs = datasets.make_blobs(n_samples=300, centers=2, cluster_std=1.0, random_state=42)

# Standardize features (essential for distance-based algorithms)
X_moons = StandardScaler().fit_transform(X_moons)
X_blobs = StandardScaler().fit_transform(X_blobs)

# Plot datasets side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c='steelblue', edgecolor='k', s=50)
axes[0].set_title("Non-linear (Moons) Dataset", fontsize=14)

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c='seagreen', edgecolor='k', s=50)
axes[1].set_title("Linear (Blobs) Dataset", fontsize=14)

plt.tight_layout()
plt.show()


## 3. Train Models (K-Means vs. DBSCAN)

We will fit both K-Means and DBSCAN on the moons dataset to show their fundamental differences.
- For **K-Means**, we'll set `n_clusters=2`.
- For **DBSCAN**, we'll set `eps=0.25` and `min_samples=5`.


In [ ]:
# Train K-Means
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
labels_kmeans = kmeans.fit_predict(X_moons)

# Train DBSCAN
dbscan = DBSCAN(eps=0.25, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_moons)


## 4. Visualize K-Means vs. DBSCAN Clustering Results

Now we'll plot the results side-by-side. 
Note that in DBSCAN, noise/outlier points are assigned a label of `-1` by scikit-learn. We will plot them as black 'X' marks to differentiate them.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot K-Means results
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_kmeans, cmap='coolwarm', edgecolor='k', s=55)
axes[0].set_title("K-Means Clustering on Moon Dataset
(Fails on non-spherical shapes)", fontsize=14)

# Plot DBSCAN results
noise_mask = (labels_dbscan == -1)

# Normal clusters (labels != -1)
axes[1].scatter(X_moons[~noise_mask, 0], X_moons[~noise_mask, 1], 
                c=labels_dbscan[~noise_mask], cmap='tab10', edgecolor='k', s=55, label='Clusters')

# Noise points (labels == -1)
axes[1].scatter(X_moons[noise_mask, 0], X_moons[noise_mask, 1], 
                c='black', marker='x', s=70, label='Noise / Outliers')

axes[1].set_title("DBSCAN Clustering on Moon Dataset
(Correctly clusters arbitrary shapes and finds noise)", fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. The Impact of Parameters ($\epsilon$ and $MinPts$)

DBSCAN is highly sensitive to its hyperparameters:
- **Epsilon ($\epsilon$)**:
  - Too small: Most points will be classified as noise because neighborhoods are too small to group together.
  - Too large: Clusters will merge into a single large cluster.
- **MinPts (min_samples)**:
  - Too small: Outliers or thin noise threads will form their own clusters.
  - Too large: Dense areas might be split or classified as noise because they don't meet the core point threshold.

Let's plot a grid showing the effect of varying both hyperparameters on the moon dataset.


In [ ]:
eps_values = [0.15, 0.25, 0.6]
min_samples_values = [3, 6, 15]

fig, axes = plt.subplots(len(eps_values), len(min_samples_values), figsize=(16, 14), sharex=True, sharey=True)

for i, eps in enumerate(eps_values):
    for j, min_samples in enumerate(min_samples_values):
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(X_moons)
        
        # Plot clusters and noise
        noise_mask = (labels == -1)
        axes[i, j].scatter(X_moons[~noise_mask, 0], X_moons[~noise_mask, 1], 
                           c=labels[~noise_mask], cmap='tab10', edgecolor='k', s=30)
        axes[i, j].scatter(X_moons[noise_mask, 0], X_moons[noise_mask, 1], 
                           c='black', marker='x', s=40)
        
        # Calculate cluster and noise counts
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = np.sum(noise_mask)
        
        axes[i, j].set_title(f"eps={eps}, min_samples={min_samples}
Clusters: {n_clusters}, Noise: {n_noise}", fontsize=11)

plt.tight_layout()
plt.show()


## 6. Finding the Optimal Epsilon (K-Distance Method)

To systematically find the optimal value for $\epsilon$:
1. Determine the value for `min_samples` (a common heuristic is to set it to $2 	imes D$, where $D$ is the dimensionality of the dataset).
2. Calculate the average distance of all points to their $k$-nearest neighbors (where $k = \text{min\_samples}$).
3. Sort these distances and plot them in ascending order.
4. The point of maximum curvature (the **elbow** point) represents where density begins to drop off significantly and is typically a very good estimate for $\epsilon$.


In [ ]:
# Since we are in 2D space, let's use k = 4 or 5
k = 5
neigh = NearestNeighbors(n_neighbors=k)
nbrs = neigh.fit(X_moons)
distances, indices = nbrs.kneighbors(X_moons)

# The distance to the k-th nearest neighbor is at index k-1 (0-indexed)
k_distances = np.sort(distances[:, k-1])

plt.figure(figsize=(10, 6))
plt.plot(k_distances, color='crimson', lw=2.5)
plt.xlabel("Points sorted by distance", fontsize=12)
plt.ylabel(f"{k}-th Nearest Neighbor Distance", fontsize=12)
plt.title(f"K-Distance Plot (k={k}) to Find Optimal Epsilon", fontsize=14)

# Draw a horizontal line indicating the elbow (where the curve sharply rises)
elbow_val = 0.23
plt.axhline(y=elbow_val, color='navy', linestyle='--', label=f'Elbow / Optimal eps (~{elbow_val})')
plt.legend(fontsize=12)
plt.show()


## 7. DBSCAN for Outlier / Anomaly Detection

A very common real-world use case for DBSCAN is anomaly detection. Since DBSCAN flags points in sparse regions as noise (`-1`), we can easily extract anomalies from normal data clusters.

Let's generate a dataset with two dense clusters and sparse background noise to test this.


In [ ]:
# Generate two standard dense blobs
X_blobs_anomaly, _ = datasets.make_blobs(n_samples=250, centers=2, cluster_std=0.55, random_state=42)

# Generate uniform random noise representing anomalies/outliers
np.random.seed(42)
anomalies = np.random.uniform(low=-4, high=4, size=(30, 2))

# Combine datasets and standardize
X_combined = np.vstack([X_blobs_anomaly, anomalies])
X_combined = StandardScaler().fit_transform(X_combined)

# Train DBSCAN with values optimized to catch outliers
dbscan_anomaly = DBSCAN(eps=0.3, min_samples=6)
labels_anomaly = dbscan_anomaly.fit_predict(X_combined)

# Plot clustering and outliers
plt.figure(figsize=(10, 6))

# Normal clusters (labeled as non-negative integers)
normal_mask = (labels_anomaly != -1)
plt.scatter(X_combined[normal_mask, 0], X_combined[normal_mask, 1], 
            c=labels_anomaly[normal_mask], cmap='viridis', edgecolor='k', s=55, label='Inliers (Clusters)')

# Anomalies (labeled as -1)
anomaly_mask = (labels_anomaly == -1)
plt.scatter(X_combined[anomaly_mask, 0], X_combined[anomaly_mask, 1], 
            c='red', marker='x', s=90, label='Anomalies / Noise')

plt.title("Using DBSCAN for Anomaly Detection", fontsize=14)
plt.legend(fontsize=12)
plt.show()
